# DeepTrace — Phase 2: data preparation & shortcut audit

**Run on:** Kaggle Notebook (free) · Accelerator **GPU T4 x2 or P100** · Internet **On**
**Inputs:** *Add Input* → search **"140k Real and Fake Faces"** (by xhlulu).
Optional cross-generator inputs (section 1b): your generated-faces dataset + a CelebA-HQ dataset.

What this notebook does, in order:
1. Index the 140k dataset into a manifest (hash, size, format, JPEG quality, EXIF) and split `valid` into `val_select` / `val_calib`.
2. **Raw audit** — can file metadata alone tell real from fake? (It must not, after preprocessing.)
3. Near-duplicate removal across splits (pHash + dHash).
4. Canonical preprocessing: strip metadata → MTCNN face crop → 224×224 → JPEG q∈[80,95].
5. **Processed audit** — the same tests again; metadata shortcuts should now be at chance (AUC ≈ 0.5).
6. (Optional) the same pipeline for the cross-generator test set.
7. Package outputs for Phase 3.

**How to run (recommended):** edit `REPO_URL` in the first cell, then **Save Version → Save & Run All
(Commit)**. It runs in the background (up to 12 h) and keeps `/kaggle/working` as the version output.
Running cells interactively works too, but files from an interactive session are not saved unless you
commit. Every script is resumable within a session.

In [ ]:
# ---- Setup: clone your repo + install the few extras Kaggle doesn't ship ----
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/<YOUR_GITHUB_USERNAME>/deeptrace.git"   # <-- EDIT ME
ON_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else Path("/content")
REPO = WORK / "deeptrace"
ML = REPO / "ml"

if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)
os.chdir(ML)
print("Working in", Path.cwd())

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements/kaggle.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements/facenet.txt", "--no-deps"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"], check=True)

import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only (prep will be slow)")

In [ ]:
# Only needed OUTSIDE Kaggle (e.g. Colab): download the dataset with the Kaggle API.
# Upload kaggle.json first (Kaggle -> Settings -> API -> Create New Token).
if not ON_KAGGLE:
    subprocess.run("pip install -q kaggle && mkdir -p ~/.kaggle && cp /content/kaggle.json ~/.kaggle/ "
                   "&& chmod 600 ~/.kaggle/kaggle.json && kaggle datasets download -d "
                   "xhlulu/140k-real-and-fake-faces -p /content/data --unzip", shell=True, check=True)
SEARCH_ROOT = Path("/kaggle/input") if ON_KAGGLE else Path("/content/data")
PROCESSED = WORK / "processed"
WORKERS = os.cpu_count() or 2

## 1. Build the 140k manifest

In [ ]:
!python scripts/build_manifest_140k.py --search-root {SEARCH_ROOT} --workers {WORKERS}

from deeptrace_ml.data.datasets import find_140k_root
ROOT_140K = find_140k_root(SEARCH_ROOT)
print("140k root:", ROOT_140K)

## 1b. (Optional) Cross-generator test set

Attach as inputs: your **generated faces** (output of `00_generate_diffusion_faces`, saved as a Kaggle
dataset) and a **CelebA-HQ** dataset (Add Input → search "CelebA-HQ"). Then set the paths below.
It is organised here so the dedupe step also checks it against the 140k data; it is prepared and audited in section 6.

⚠️ CelebA-HQ images were created with a neural JPEG-artifact-removal + 4× super-resolution step
(Karras et al., 2018), so these "real" images carry some neural processing. Keep that in mind when a
detector calls them fake — it's documented as a known confound.

In [ ]:
CELEBAHQ_DIR = None        # e.g. Path("/kaggle/input/<celebahq-dataset>/<images-folder>")
GENERATED_DIR = None       # e.g. Path("/kaggle/input/deeptrace-generated")  (contains sdxl/ and flux-schnell/)
PER_SOURCE = 1000

if CELEBAHQ_DIR and GENERATED_DIR:
    sources = ["--source", f"celebahq:real:{CELEBAHQ_DIR}"]
    for name, folder in (("sdxl", "sdxl"), ("flux_schnell", "flux-schnell")):
        if (GENERATED_DIR / folder).exists():
            sources += ["--source", f"{name}:fake:{GENERATED_DIR / folder}"]
    subprocess.run([sys.executable, "scripts/organize_external.py", "--dataset", "crossgen",
                    "--out-root", str(WORK / "raw/crossgen"), *sources,
                    "--max-per-source", str(PER_SOURCE), "--workers", str(WORKERS)], check=True)
else:
    print("Skipping cross-generator set (paths not set).")

## 2. Raw audit (before any preprocessing)

How to read the results:
- **Metadata classifier AUC** near 0.5 = no shortcut; near 1.0 = the label is leaking through file properties.
- **Pixel-statistics AUC** can legitimately be above 0.5 (generators do leave colour/texture traces), but a very high value warns that a model could win with crude rules like "smoother = fake".
- **Spectra:** look for bright off-centre dots or grid lines (upsampling artifacts) and a bump in the tail of the radial profile.

In [ ]:
!python scripts/run_audit.py --dataset 140k --manifest data/manifests/140k_raw.csv.gz --root 140k={ROOT_140K} --workers {WORKERS}

In [ ]:
from IPython.display import Image as ShowImage, display
PLOTS = REPO / "docs/plots/audit/140k"
for name in ["raw_metadata_numeric", "raw_metadata_flags", "raw_pixel_stats", "raw_spectra_1d", "raw_spectra_2d"]:
    print(name); display(ShowImage(filename=str(PLOTS / f"{name}.png")))

## 3. Near-duplicates across splits

In [ ]:
MANIFESTS = ["--manifest", "data/manifests/140k_raw.csv.gz", "--root", f"140k={ROOT_140K}"]
if Path("data/manifests/crossgen_raw.csv.gz").exists():          # created in section 1b
    MANIFESTS += ["--manifest", "data/manifests/crossgen_raw.csv.gz", "--root", f"crossgen={WORK / 'raw/crossgen'}"]
subprocess.run([sys.executable, "scripts/dedupe.py", *MANIFESTS, "--workers", str(WORKERS)], check=True)

import json
print(json.dumps(json.load(open("reports/dedupe_report.json")), indent=2))
dup_plot = REPO / "docs/plots/audit/near_duplicates.png"
if dup_plot.exists():
    display(ShowImage(filename=str(dup_plot)))   # eyeball: are these really the same face?

## 4. Canonical preprocessing (GPU)

MTCNN runs on the GPU in batches. Progress is saved after every batch, so a timeout loses nothing —
rerun the cell. Speed depends on the GPU; the script prints throughput as it goes.

In [ ]:
!python scripts/prepare_dataset.py --manifest data/manifests/140k_raw.csv.gz --root 140k={ROOT_140K} \
    --duplicates data/manifests/duplicates_drop.csv --out-root {PROCESSED} \
    --device auto --batch-size 128 --io-workers {WORKERS}

print(json.dumps(json.load(open("reports/prepare_140k_raw.json"))["status_counts"], indent=2))

## 5. Processed audit — did preprocessing remove the shortcuts?

In [ ]:
!python scripts/run_audit.py --dataset 140k --manifest data/manifests/140k_raw.csv.gz --root 140k={ROOT_140K} \
    --processed-manifest data/manifests/140k_processed.csv.gz --processed-root {PROCESSED} --workers {WORKERS}

for name in ["processed_status", "processed_metadata_numeric", "processed_pixel_stats", "processed_spectra_1d", "processed_spectra_2d"]:
    print(name); display(ShowImage(filename=str(PLOTS / f"{name}.png")))

In [ ]:
# Before/after table (the numbers that go into the README — straight from the JSON, never retyped)
import pandas as pd
audit = json.load(open("reports/audit_140k.json"))
rows = []
for stage in ("raw", "processed"):
    for key, res in audit[stage].items():
        if key.startswith("trivial_classifier"):
            rows.append({"stage": stage, "features": res["feature_set"],
                         "logreg_auc": round(res["auc"]["logistic_regression"]["mean"], 3),
                         "boosting_auc": round(res["auc"]["gradient_boosting"]["mean"], 3),
                         "top_features": ", ".join(f["feature"] for f in res["top_features"][:3])})
pd.DataFrame(rows)

## 6. (Optional) Prepare & audit the cross-generator set

In [ ]:
if Path("data/manifests/crossgen_raw.csv.gz").exists():
    CROSS_ROOT = WORK / "raw/crossgen"
    subprocess.run([sys.executable, "scripts/prepare_dataset.py", "--manifest", "data/manifests/crossgen_raw.csv.gz",
                    "--root", f"crossgen={CROSS_ROOT}", "--duplicates", "data/manifests/duplicates_drop.csv",
                    "--out-root", str(PROCESSED), "--device", "auto", "--batch-size", "32"], check=True)
    subprocess.run([sys.executable, "scripts/run_audit.py", "--dataset", "crossgen",
                    "--manifest", "data/manifests/crossgen_raw.csv.gz", "--root", f"crossgen={CROSS_ROOT}",
                    "--processed-manifest", "data/manifests/crossgen_processed.csv.gz",
                    "--processed-root", str(PROCESSED), "--sample-per-group", "1000",
                    "--spectrum-per-group", "500", "--workers", str(WORKERS)], check=True)
    CROSS_PLOTS = REPO / "docs/plots/audit/crossgen"
    for name in ["raw_metadata_numeric", "processed_status", "processed_spectra_1d"]:
        print(name); display(ShowImage(filename=str(CROSS_PLOTS / f"{name}.png")))
else:
    print("No cross-generator manifest — skipped.")

## 7. Save outputs

1. If you ran interactively, commit with **Save Version → Save & Run All** (only committed runs keep files).
2. Open the finished version → **Output** → **New Dataset**, name it `deeptrace-processed`.
   Phase 3 training attaches this dataset (no re-processing needed).
3. Download `deeptrace_phase2_artifacts.zip` (below) and commit its manifests, reports and plots to
   your repo — those are small and make the audit reproducible and reviewable on GitHub.

In [ ]:
import shutil
bundle = WORK / "deeptrace_phase2_artifacts"
if bundle.exists():
    shutil.rmtree(bundle)
shutil.copytree(ML / "data/manifests", bundle / "ml/data/manifests")
shutil.copytree(ML / "reports", bundle / "ml/reports")
shutil.copytree(REPO / "docs/plots", bundle / "docs/plots")
print(shutil.make_archive(str(bundle), "zip", bundle))